In [ ]:
# !pip install earthpy gdal
!pip install --upgrade pip
!apt-get install -y gdal-bin
!pip install gdal


In [2]:
from osgeo import gdal, gdal_array
import os
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from skimage import io
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torch.optim as optim

mask_path="/kaggle/input/cloud-masking-dataset/content/train/masks"
data_path ="/kaggle/input/cloud-masking-dataset/content/train/data"



ModuleNotFoundError: No module named 'osgeo'

In [ ]:
def display_image(file_name,data_only):
    file_path=os.path.join(data_path, file_name)
    img_ds= gdal.Open(file_path, gdal.GA_ReadOnly)
    num_bands=img_ds.RasterCount
    if( not data_only):
        fig, axs = plt.subplots(1, 5, figsize=(10, 5))
    else:
        fig, axs = plt.subplots(1, 4, figsize=(10, 5))
    
    for i in range(1,num_bands+1):
      band = img_ds.GetRasterBand(i)
      img = band.ReadAsArray()
      axs[i-1].imshow(img, cmap='gray')
      axs[i-1].set_title('Original Image')
    if(not data_only):
        file_path=os.path.join(mask_path, file_name)
        img_ds= gdal.Open(file_path, gdal.GA_ReadOnly)
        num_bands=img_ds.RasterCount
        band = img_ds.GetRasterBand(1)
        img = band.ReadAsArray()
        axs[4].imshow(img, cmap='gray', vmin=0, vmax=1)        
        axs[4].set_title('Mask')
    plt.show()

***Display Random Images***

In [ ]:

data= os.listdir(data_path)

train_data, val_test = train_test_split(data, test_size=0.2, random_state=42)
val, test = train_test_split(val_test, test_size=0.5, random_state=42)

random_indices = np.random.permutation(len(train_data))[:10]
for i in random_indices:
    file_name=train_data[i]
    display_image(file_name,False)

***Show Correlation Between Bands***

In [ ]:
subset = random.sample(train_data, k=1000)
all_pixels = []
for file_path in tqdm(subset):
    file_path=os.path.join(data_path, file_path)
    img_ds = gdal.Open(file_path, gdal.GA_ReadOnly)
    width = img_ds.RasterXSize
    height = img_ds.RasterYSize
    num_bands = img_ds.RasterCount
    img_array = np.zeros((height, width, num_bands), dtype=np.float32)
    for i in range(num_bands):
        band = img_ds.GetRasterBand(i + 1)
        img_array[:, :, i] = band.ReadAsArray()
    reshaped = img_array.reshape(-1, 4)
    all_pixels.append(reshaped)

combined = np.vstack(all_pixels)  # Shape: (total_pixels, 4)
corr = np.corrcoef(combined.T)
sns.heatmap(corr, annot=True, cmap='coolwarm', xticklabels=['B1', 'B2', 'B3', 'B4'], yticklabels=['B1', 'B2', 'B3', 'B4'])
plt.title("Global Band Correlation Across 1000 Images")
plt.show()









***Apply PCA On Bands 3&4***

In [ ]:
# subset = random.sample(train_data, k=1000)
# images = []
# for fname in subset:
#     img_path = os.path.join(data_path, fname)
#     img_ds = gdal.Open(img_path, gdal.GA_ReadOnly)
#     for i in [3,4]:
#         band = img_ds.GetRasterBand(i)
#         img_array[:, :, i-3] = band.ReadAsArray()
#     reshaped = img_array.reshape(-1, 2)
#     images.append(reshaped)


# X = np.vstack(images)  # Shape: (total_pixels, 2)

# # Scale the data to have zero mean and unit variance along each feature axis
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# # Perform PCA with 3 components
# pca = PCA()
# pca.fit(X_scaled)
# output = pca.transform(X_scaled)

# # Scree plot of the eigen value ratios
# explained_variance_ratio = pca.explained_variance_ratio_
# plt.figure()
# plt.plot(explained_variance_ratio, marker='o', linestyle='--')
# plt.xlabel('Principal Component')
# plt.ylabel('Explained Variance Ratio')
# plt.title('Explained Variance Ratio of Principal Components')
# plt.show()

# # Plot the 3D graph of the PCA (scatter plot of first 3 principal components)
# pc1 = output[:, 0]




# Save the fitted PCA model
# joblib.dump(pca, 'pca_model.joblib')
# joblib.dump(scaler, 'scaler_model.joblib')


# === Load Saved Models ===
pca = joblib.load('/kaggle/input/pca/other/default/1/pca_model.joblib')
scaler = joblib.load('/kaggle/input/pca/other/default/1/scaler_model.joblib')



In [ ]:
def apply_pca(file_name,is_file_name):
    # === Select and Load a New Image ===
    if(is_file_name):
        new_image_path = os.path.join(data_path, file_name)  # pick one test image
    else:
        new_image_path=file_name
        
    img_ds = gdal.Open(new_image_path, gdal.GA_ReadOnly)
    
    # === Read Band 3 and 4 ===
    band3 = img_ds.GetRasterBand(3).ReadAsArray()
    band4 = img_ds.GetRasterBand(4).ReadAsArray()
    
    # === Stack and Reshape ===
    img_array = np.stack([band3, band4], axis=-1)  # shape: (H, W, 2)
    h, w, _ = img_array.shape
    reshaped = img_array.reshape(-1, 2)
    
    # === Transform with Scaler and PCA ===
    scaled = scaler.transform(reshaped)
    pca_output = pca.transform(scaled)
    
    # === Select First Principal Component and Reshape to Image ===
    pc1 = pca_output[:, 0].reshape(h, w)

    # plt.figure(figsize=(10, 6))
    # plt.imshow(pc1, cmap='gray')
    # plt.colorbar()
    # plt.title('First Principal Component of New Image')
    # plt.axis('off')
    # plt.show()

    return pc1



***Show Counts***

In [ ]:
fully_cloud=0
partially_cloud=0
cloud_free=0
for i,file_name in tqdm(enumerate(train_data)):
    file_path=os.path.join(mask_path, file_name)
    img_ds= gdal.Open(file_path, gdal.GA_ReadOnly)
    band = img_ds.GetRasterBand(1)
    img = band.ReadAsArray()
    if (np.all(img == 0)):
        cloud_free+=1
        if(cloud_free%200==0):
            display_image(file_name,False)
    elif (np.all(img == 1)):
        fully_cloud+=1
        if(fully_cloud%200==0):
             display_image(file_name,False)
    else:
        partially_cloud+=1
# 📊 Plotting
labels = ['Cloud-Free', 'Fully Cloudy', 'Partially Cloudy']
counts = [cloud_free, fully_cloud, partially_cloud]
plt.figure(figsize=(8, 5))
bars = plt.bar(labels, counts, color=['skyblue', 'lightcoral', 'lightgreen'])
plt.title('Mask Category Distribution')
plt.ylabel('Number of Images')
plt.xlabel('Category')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 5, int(yval), ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:


class CloudMaskDataset(Dataset):
    def __init__(self,image_files):
        self.image_paths = [os.path.join(data_path, fname) for fname in image_files]
        self.mask_paths = [os.path.join(mask_path, fname) for fname in image_files]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img=apply_pca(self.image_paths [idx],False)
        mask_data= gdal.Open(self.mask_paths[idx], gdal.GA_ReadOnly)
        band = mask_data.GetRasterBand(1)
        mask = band.ReadAsArray()
        

        # # Normalize to [0, 1]
        # img = img.astype(np.float32) / 255.0
        # mask = mask.astype(np.float32) / 255.0

        # Expand channel dimension: [H, W] → [1, H, W]
        img = np.expand_dims(img, axis=0)
        mask = np.expand_dims(mask, axis=0)

        return torch.from_numpy(img), torch.from_numpy(mask)


In [ ]:


class DoubleConv(nn.Module):
    """(conv => BN => ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()

        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        d4 = self.down4(p3)
        p4 = self.pool4(d4)

        # Bottleneck
        bn = self.bottleneck(p4)

        # Decoder
        up4 = self.up4(bn)
        cat4 = torch.cat([up4, d4], dim=1)
        d4 = self.conv4(cat4)

        up3 = self.up3(d4)
        cat3 = torch.cat([up3, d3], dim=1)
        d3 = self.conv3(cat3)

        up2 = self.up2(d3)
        cat2 = torch.cat([up2, d2], dim=1)
        d2 = self.conv2(cat2)

        up1 = self.up1(d2)
        cat1 = torch.cat([up1, d1], dim=1)
        d1 = self.conv1(cat1)

        out = self.final_conv(d1)
        return torch.sigmoid(out)  # For binary mask output




In [ ]:
def train(model, loader, optimizer, criterion):
    model.train()
    epoch_loss = 0
    for imgs, masks in tqdm(loader,desc="training loop"):
        print(torch.cuda.memory_summary())
        torch.cuda.empty_cache()
        imgs = imgs.to(torch.float32).to(device)
        masks = masks.to(torch.float32).to(device)
        outputs = model(imgs)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    return epoch_loss / len(loader)

In [ ]:
def dice_coefficient(preds, targets, smooth=1e-6):
    preds = torch.sigmoid(preds)  # Convert logits to probabilities
    preds = (preds > 0.5).float()  # Threshold to binary mask

    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))

    dice = (2. * intersection + smooth) / (union + smooth)
    return dice.mean()


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    epoch_dice = 0
    with torch.no_grad():
        for imgs, masks in tqdm(loader,desc="testing loop"):
            imgs = imgs.to(device)
            masks = masks.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, masks)
            dice = dice_coefficient(outputs, masks)

            epoch_loss += loss.item()
            epoch_dice += dice.item()

    avg_loss = epoch_loss / len(loader)
    avg_dice = epoch_dice / len(loader)
    return avg_loss, avg_dice


In [ ]:
print(train_data[:10])

In [ ]:


# Create dataset and dataloader for train_data
train_dataset = CloudMaskDataset(train_data)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True,num_workers=2)

# Create dataset and dataloader for validation
val_dataset = CloudMaskDataset(val)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True,num_workers=2)

# Create dataset and dataloader tset
test_dataset = CloudMaskDataset(test)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True,num_workers=2)




In [ ]:

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize model and move to device
model = UNet(in_channels=1, out_channels=1).to(device)

# Define loss and optimizer
criterion = nn.BCEWithLogitsLoss()  # For binary segmentation
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
epochs=10
for epoch in range(epochs+1):
    print(torch.cuda.memory_summary())
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss, val_dice = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")
